<a href="https://colab.research.google.com/github/Aryan-0042/Data-Science-/blob/main/BE_P_Assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
"""
================================================================================
LAB: Student Record Management System
File: Day1_Lab_RecordManager_starter.py
--------------------------------------------------------------------------------
Trainer Section (First 30%):
  - In-memory database initialization
  - File loading logic with JSON exception handling
  - View all records formatted output
  - Interactive CLI loop scaffold

Student Completion Tasks (Remaining 70%):
  1. Complete add_student_record() with input validation
  2. Implement search_student_record() by ID or Name substring
  3. Implement delete_student_record() with confirmation
  4. Implement update_student_record()
  5. Implement save_records_to_json() with safe file flushing
  6. STRETCH GOAL: Implement export_to_csv()
================================================================================
"""

import json
import os
import sys
import csv  # Added for CSV export
from typing import Dict, Any

# Target data file
DATABASE_FILE = "sample_records.json"

# In-memory storage: Key = Student ID, Value = Dict of Student attributes
STUDENT_REGISTRY: Dict[str, Dict[str, Any]] = {}


def load_records_from_json(file_path: str) -> Dict[str, Dict[str, Any]]:
    """Loads student records safely from a JSON file.

    Handles FileNotFoundError and corrupted JSON formatting gracefully.
    """
    if not os.path.exists(file_path):
        print(f"[WARN] Database file '{file_path}' not found. Starting with empty registry.")
        return {}

    try:
        with open(file_path, "r", encoding="utf-8") as file:
            data = json.load(file)
            print(f"[SUCCESS] Loaded {len(data)} record(s) from {file_path}.")
            return data
    except json.JSONDecodeError as json_err:
        print(f"[ERROR] Corrupted JSON structure in '{file_path}': {json_err}")
        return {}
    except Exception as err:
        print(f"[UNEXPECTED ERROR] Failed to load data: {err}")
        return {}


def view_all_records(registry: Dict[str, Dict[str, Any]]) -> None:
    """Prints all student records in a formatted tabular view."""
    if not registry:
        print("\n[INFO] No records found in the registry.")
        return

    separator = "-" * 75
    print("\n" + separator)
    print(f"{'Student ID':<12} | {'Name':<22} | {'Branch':<22} | {'CGPA':<5}")
    print(separator)
    for student_id, details in registry.items():
        name = details.get("name", "N/A")
        branch = details.get("branch", "N/A")
        cgpa = details.get("cgpa", 0.0)
        print(f"{student_id:<12} | {name:<22} | {branch:<22} | {cgpa:<5.2f}")
    print(separator + "\n")


# ==============================================================================
# ✍️ STUDENT TASKS COMPLETED BELOW
# ==============================================================================

def add_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """Task 1: Prompt user for student ID, name, branch, and CGPA."""
    print("\n--- Add New Student ---")

    student_id = input("Enter Student ID: ").strip()
    if not student_id:
        print("[ERROR] Student ID cannot be empty.")
        return
    if student_id in registry:
        print(f"[ERROR] Student ID '{student_id}' already exists.")
        return

    name = input("Enter Student Name: ").strip()
    if not name:
        print("[ERROR] Name cannot be empty.")
        return

    branch = input("Enter Branch: ").strip()
    if not branch:
        print("[ERROR] Branch cannot be empty.")
        return

    try:
        cgpa = float(input("Enter CGPA (0.0 - 10.0): "))
        if not (0.0 <= cgpa <= 10.0):
            print("[ERROR] CGPA must be a valid number between 0.0 and 10.0.")
            return
    except ValueError:
        print("[ERROR] Invalid input. CGPA must be a decimal number.")
        return

    # Auto-generate email
    first_name = name.split()[0].lower()
    email = f"{first_name}.{student_id.lower()}@university.edu"

    # Add to registry
    registry[student_id] = {
        "name": name,
        "branch": branch,
        "cgpa": cgpa,
        "email": email
    }
    print(f"[SUCCESS] Student '{name}' added successfully! Auto-generated email: {email}")


def search_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """Task 2: Search by student ID (exact) or student Name (case-insensitive substring)."""
    print("\n--- Search Student Records ---")
    if not registry:
        print("[INFO] No records available to search.")
        return

    query = input("Enter Student ID or Name to search: ").strip().lower()
    found = False

    for student_id, details in registry.items():
        if query == student_id.lower() or query in details.get("name", "").lower():
            print(f"\n[MATCH FOUND] Student ID: {student_id}")
            for key, value in details.items():
                print(f"  - {key.capitalize()}: {value}")
            found = True

    if not found:
        print(f"[INFO] No student records found matching '{query}'.")


def delete_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """Task 3: Prompt for Student ID and delete record with confirmation."""
    print("\n--- Delete Student Record ---")
    student_id = input("Enter Student ID to delete: ").strip()

    if student_id in registry:
        student_name = registry[student_id].get("name", "Unknown")
        confirm = input(f"Are you sure you want to delete the record for {student_name}? (Y/N): ").strip().upper()
        if confirm == 'Y':
            del registry[student_id]
            print(f"[SUCCESS] Record for Student ID '{student_id}' has been deleted.")
        else:
            print("[INFO] Deletion cancelled.")
    else:
        print(f"[ERROR] Student ID '{student_id}' not found.")


def save_records_to_json(file_path: str, registry: Dict[str, Dict[str, Any]]) -> None:
    """Task 4: Serialize the in-memory registry dictionary to the JSON file safely."""
    try:
        with open(file_path, "w", encoding="utf-8") as file:
            json.dump(registry, file, indent=2)
        print(f"[SUCCESS] Successfully saved {len(registry)} record(s) to '{file_path}'.")
    except Exception as err:
        print(f"[ERROR] Failed to save data to JSON: {err}")


def export_to_csv(file_path: str, registry: Dict[str, Dict[str, Any]]) -> None:
    """Task 5 (Bonus): Export all student records to a CSV file."""
    if not registry:
        print("[INFO] No records available to export.")
        return

    try:
        with open(file_path, mode="w", newline="", encoding="utf-8") as file:
            # Determine headers from the first record
            sample_student = next(iter(registry.values()))
            fieldnames = ["student_id"] + list(sample_student.keys())

            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()

            for student_id, details in registry.items():
                row = {"student_id": student_id}
                row.update(details)
                writer.writerow(row)

        print(f"[SUCCESS] Successfully exported {len(registry)} record(s) to '{file_path}'.")
    except Exception as err:
        print(f"[ERROR] Failed to export data to CSV: {err}")


def main_menu() -> None:
    """Main CLI control loop."""
    global STUDENT_REGISTRY
    STUDENT_REGISTRY = load_records_from_json(DATABASE_FILE)

    menu_banner = """
========================================
🎓 STUDENT RECORD MANAGEMENT SYSTEM
========================================
1. View All Records
2. Add Student Record
3. Search Record
4. Delete Record
5. Save Database to File
6. Export Records to CSV (Bonus)
0. Save & Exit
========================================
"""
    while True:
        print(menu_banner)
        choice = input("Enter choice [0-6]: ").strip()

        if choice == "1":
            view_all_records(STUDENT_REGISTRY)
        elif choice == "2":
            add_student_record(STUDENT_REGISTRY)
        elif choice == "3":
            search_student_record(STUDENT_REGISTRY)
        elif choice == "4":
            delete_student_record(STUDENT_REGISTRY)
        elif choice == "5":
            save_records_to_json(DATABASE_FILE, STUDENT_REGISTRY)
        elif choice == "6":
            export_to_csv("students_export.csv", STUDENT_REGISTRY)
        elif choice == "0":
            save_records_to_json(DATABASE_FILE, STUDENT_REGISTRY)
            print("[INFO] Application closed successfully. Good bye!")
            break
        else:
            print("[WARN] Invalid option selected. Please enter a number between 0 and 6.")


if __name__ == "__main__":
    main_menu()

[WARN] Database file 'sample_records.json' not found. Starting with empty registry.

🎓 STUDENT RECORD MANAGEMENT SYSTEM
1. View All Records
2. Add Student Record
3. Search Record
4. Delete Record
5. Save Database to File
6. Export Records to CSV (Bonus)
0. Save & Exit

Enter choice [0-6]: 2

--- Add New Student ---
Enter Student ID: 2023215705
Enter Student Name: Aryan
Enter Branch: B.Tech CSE
Enter CGPA (0.0 - 10.0): 8.6
[SUCCESS] Student 'Aryan' added successfully! Auto-generated email: aryan.2023215705@university.edu

🎓 STUDENT RECORD MANAGEMENT SYSTEM
1. View All Records
2. Add Student Record
3. Search Record
4. Delete Record
5. Save Database to File
6. Export Records to CSV (Bonus)
0. Save & Exit

Enter choice [0-6]: 1

---------------------------------------------------------------------------
Student ID   | Name                   | Branch                 | CGPA 
---------------------------------------------------------------------------
2023215705   | Aryan                  | B.